# NB_LIBRO_02 · Figuras del capítulo 3 (Calidad y preprocesamiento)

**Libro:** *AgroVisión: del píxel a la decisión*

**Usa (solo lectura):** `PALMA_AFRICANA_YOLO/02_DATOS_PROCESADOS/classification_5class_v1` (452 imágenes).

**Produce en `AGROVISION_LIBRO/`:** las figuras 3.1 a 3.7 en `02_FIGURAS/cap03/`, y en `03_VALORES/` el archivo `valores_cap03.json` y tres tablas CSV.

**Cómo ejecutarlo:** *Entorno de ejecución → Ejecutar todo*. No necesita GPU. Tarda unos 5 minutos. Después ejecute `NB_LIBRO_03_Figuras_Capitulo_4`.

## 1. Configuración

In [ ]:
import os
import io
import re
import json
import math
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Carpeta raíz de Drive. Fuera de Colab puede indicarse con la variable AGROVISION_BASE.
BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))

DIR_PALMA = BASE / 'PALMA_AFRICANA_YOLO' / '02_DATOS_PROCESADOS' / 'classification_5class_v1'

DIR_LIBRO = BASE / 'AGROVISION_LIBRO'
DIR_FIG3 = DIR_LIBRO / '02_FIGURAS' / 'cap03'
DIR_VAL = DIR_LIBRO / '03_VALORES'
for d in (DIR_FIG3, DIR_VAL):
    d.mkdir(parents=True, exist_ok=True)

# --- Parámetros que puede cambiar -------------------------------------------
# Imagen de palma usada como ejemplo. Con None se usa la misma del capítulo 2
# (se lee de 03_VALORES/valores_para_texto.json).
IMG_MADURO = None
SEMILLA = 2026
# -----------------------------------------------------------------------------

assert DIR_PALMA.exists(), f'No se encuentra el dataset de palma en: {DIR_PALMA}'
print('Dataset de palma :', DIR_PALMA)
print('Salida           :', DIR_LIBRO)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)


import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

LADO_CAL = 512        # lado mayor al que se llevan las imágenes antes de medir su calidad
RNG = np.random.default_rng(SEMILLA)
BORDE_BLANCO = [pe.Stroke(linewidth=2.0, foreground='#0b0b0b'), pe.Normal()]


def miles(x):
    # Entero con espacio como separador de miles.
    return f'{x:,.0f}'.replace(',', ' ')


def elemento(k):
    # Elemento estructurante elíptico de k × k píxeles.
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def rotulo_fila(ax, texto):
    ax.set_ylabel(texto, fontsize=8, color=TINTA)

## 2. Inventario del dataset de palma e imagen de ejemplo

In [ ]:
filas_inv = []
for ruta in sorted(DIR_PALMA.rglob('*')):
    if not ruta.is_file() or ruta.suffix.lower() not in EXT_IMG:
        continue
    partes = [p.lower() for p in ruta.relative_to(DIR_PALMA).parts[:-1]]
    clase = next((p for p in partes if p in CLASES), None)
    particion = next((p for p in partes if p in ('train', 'val', 'test')), 'sin_split')
    if clase is None:
        continue
    filas_inv.append({'ruta': str(ruta), 'archivo': ruta.name, 'split': particion, 'clase': clase})
inv = pd.DataFrame(filas_inv)
assert len(inv) > 0, 'No se encontraron imágenes. Revise DIR_PALMA.'
print(inv.groupby(['clase', 'split']).size().unstack(fill_value=0).reindex(CLASES).rename(index=NOMBRE_ES).to_string())
print('Total de imágenes:', len(inv))

# Imagen de ejemplo: la misma del capítulo 2, salvo que se indique otra.
nombre_ej = IMG_MADURO
ruta_val01 = DIR_VAL / 'valores_para_texto.json'
if nombre_ej is None and ruta_val01.exists():
    with open(ruta_val01, encoding='utf-8') as fh:
        nombre_ej = json.load(fh).get('imagenes_ejemplo', {}).get('maduro', {}).get('archivo')
hit = inv[inv['archivo'] == nombre_ej] if nombre_ej else inv.iloc[0:0]
if len(hit) == 0:
    hit = inv[(inv['clase'] == 'ripe') & (inv['split'] == 'train')]
    print('AVISO: no se encontró la imagen de ejemplo del capítulo 2; se usa la primera de la clase maduro.')
fila_ej = hit.iloc[0]
img_ej_full = leer_rgb(fila_ej['ruta'])
img_ej = redimensionar(img_ej_full, LADO_CAL) if max(img_ej_full.shape[:2]) > LADO_CAL else img_ej_full
print('Imagen de ejemplo:', fila_ej['archivo'], img_ej_full.shape[1], '×', img_ej_full.shape[0])
VALORES['imagen_ejemplo_palma'] = {'archivo': fila_ej['archivo'], 'ancho': int(img_ej_full.shape[1]),
                                   'alto': int(img_ej_full.shape[0])}

# Capítulo 3 · Calidad y preprocesamiento

## 3. Degradaciones y métricas de calidad

Las siete degradaciones y sus dos niveles son los mismos de la prueba de robustez del proyecto de palma (NB07). Las métricas de calidad se calculan sobre la imagen en gris, con el lado mayor reducido a 512 píxeles:

- **brillo**: media de los niveles de gris;
- **contraste**: desviación estándar de los niveles de gris;
- **entropía**: entropía de Shannon del histograma, en bits;
- **nitidez**: varianza del laplaciano;
- **píxeles saturados**: porcentaje con nivel ≤ 5 o ≥ 250.

In [ ]:
# (moderado, severo), como en NB07 del proyecto de palma.
PARAM_DEG = {
    'oscurecimiento': (0.75, 0.55),      # factor multiplicativo
    'sobreexposicion': (1.25, 1.45),     # factor multiplicativo
    'contraste': (0.70, 0.45),           # factor alrededor de la media
    'desenfoque': (1.0, 2.5),            # sigma del filtro gaussiano, en píxeles
    'ruido': (12, 28),                   # desviación estándar del ruido gaussiano
    'rotacion': (8, 18),                 # grados
    'jpeg': (60, 30),                    # calidad JPEG
}
NOMBRE_DEG = {'oscurecimiento': 'Oscurecimiento', 'sobreexposicion': 'Sobreexposición', 'contraste': 'Contraste bajo',
              'desenfoque': 'Desenfoque', 'ruido': 'Ruido', 'rotacion': 'Rotación', 'jpeg': 'Compresión JPEG'}
# F1 macro de YOLO11m (semilla 2026) en las 70 imágenes de prueba, según NB07.
F1_NB07 = {('limpia', ''): 0.867, ('oscurecimiento', 'severo'): 0.851, ('sobreexposicion', 'severo'): 0.856,
           ('contraste', 'severo'): 0.670, ('desenfoque', 'moderado'): 0.781, ('desenfoque', 'severo'): 0.506,
           ('ruido', 'moderado'): 0.906, ('ruido', 'severo'): 0.778, ('rotacion', 'severo'): 0.814,
           ('jpeg', 'severo'): 0.874}


def degradar(img, tipo, valor, semilla=0):
    f = img.astype(np.float32)
    if tipo in ('oscurecimiento', 'sobreexposicion'):
        sal = f * valor
    elif tipo == 'contraste':
        media = f.mean()
        sal = (f - media) * valor + media
    elif tipo == 'desenfoque':
        sal = cv2.GaussianBlur(f, (0, 0), valor)
    elif tipo == 'ruido':
        sal = f + np.random.default_rng(semilla).normal(0, valor, f.shape)
    elif tipo == 'rotacion':
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), valor, 1.0)
        sal = cv2.warpAffine(f, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    elif tipo == 'jpeg':
        buf = io.BytesIO()
        Image.fromarray(img).save(buf, 'JPEG', quality=int(valor))
        return np.asarray(Image.open(io.BytesIO(buf.getvalue())).convert('RGB'))
    else:
        raise ValueError(tipo)
    return np.clip(sal, 0, 255).astype(np.uint8)


def metricas_calidad(rgb):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    p = np.bincount(g.ravel(), minlength=256) / g.size
    p = p[p > 0]
    return {'brillo': float(g.mean()), 'contraste': float(g.std()),
            'entropia': float(-(p * np.log2(p)).sum()),
            'nitidez': float(cv2.Laplacian(g, cv2.CV_64F).var()),
            'pct_oscuros': float((g <= 5).mean() * 100), 'pct_saturados': float((g >= 250).mean() * 100)}


def texto_param(tipo, valor):
    if tipo in ('oscurecimiento', 'sobreexposicion', 'contraste'):
        return f'factor {coma(valor, 2)}'
    if tipo == 'desenfoque':
        return f'σ = {coma(valor, 1)} px'
    if tipo == 'ruido':
        return f'σ = {valor} niveles'
    if tipo == 'rotacion':
        return f'{valor}°'
    return f'calidad {valor}'

### Figura 3.1 · Una imagen y siete degradaciones (nivel severo)

In [ ]:
orden_deg = ['oscurecimiento', 'sobreexposicion', 'contraste', 'desenfoque', 'ruido', 'rotacion', 'jpeg']
prop = img_ej.shape[0] / img_ej.shape[1]
fig, axs = plt.subplots(2, 4, figsize=(ANCHO_IN, ANCHO_IN * prop / 4 * 2 + 0.75), constrained_layout=True)
paneles = [('Original', None, None)] + [(NOMBRE_DEG[t], t, PARAM_DEG[t][1]) for t in orden_deg]
tabla_deg = [{'condicion': 'Original', 'nivel': '', 'parametro': '', **metricas_calidad(img_ej)}]
for ax, letra, (nombre, tipo, valor) in zip(axs.ravel(), 'abcdefgh', paneles):
    vista = img_ej if tipo is None else degradar(img_ej, tipo, valor)
    ax.imshow(vista); sin_ejes(ax)
    sub = '' if tipo is None else '\n' + texto_param(tipo, valor)
    ax.set_title(f'({letra}) {nombre}{sub}', linespacing=1.15, fontsize=7.5)
for t in orden_deg:
    for nivel, valor in zip(('moderado', 'severo'), PARAM_DEG[t]):
        tabla_deg.append({'condicion': NOMBRE_DEG[t], 'nivel': nivel, 'parametro': texto_param(t, valor),
                          **metricas_calidad(degradar(img_ej, t, valor))})
guardar(fig, DIR_FIG3 / 'Fig_3_1_degradaciones.png')
plt.show()
tabla_deg = pd.DataFrame(tabla_deg)
tabla_deg.round(2).to_csv(DIR_VAL / 'tabla_3_metricas_imagen_degradada.csv', index=False)
print(tabla_deg.round(1).to_string(index=False))
VALORES['metricas_imagen_ejemplo_degradada'] = json.loads(tabla_deg.round(2).to_json(orient='records', force_ascii=False))

### Métricas de calidad de las 452 imágenes

Se guarda una fila por imagen en `03_VALORES/calidad_por_imagen.csv`. Las imágenes reducidas quedan en memoria para los pasos siguientes.

In [ ]:
IMG_CAL = {}                    # ruta -> imagen RGB con lado mayor de 512 px
registros_cal = []
for k, fila in enumerate(inv.itertuples(index=False), start=1):
    im = leer_rgb(fila.ruta, lado_max=LADO_CAL)
    IMG_CAL[fila.ruta] = im
    registros_cal.append(metricas_calidad(im))
    if k % 50 == 0 or k == len(inv):
        print(f'  {k}/{len(inv)} imágenes medidas')
cal = pd.concat([inv.reset_index(drop=True), pd.DataFrame(registros_cal)], axis=1)
cal.drop(columns=['ruta']).round(3).to_csv(DIR_VAL / 'calidad_por_imagen.csv', index=False)

METRICAS = ['brillo', 'contraste', 'entropia', 'nitidez']
perc = cal[METRICAS + ['pct_oscuros', 'pct_saturados']].quantile([0.01, 0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975, 0.99])
perc.index = ['P1', 'P2,5', 'P5', 'P25', 'P50', 'P75', 'P95', 'P97,5', 'P99']
print(perc.round(2).to_string())
por_clase_cal = cal.groupby('clase')[METRICAS].median().reindex(CLASES).rename(index=NOMBRE_ES)
print('\nMediana por clase:'); print(por_clase_cal.round(1).to_string())

# Umbrales de partida de la regla de aceptación (sección 3.5): percentiles del propio conjunto.
UMBRAL = {'nitidez_min': float(perc.loc['P5', 'nitidez']), 'contraste_min': float(perc.loc['P5', 'contraste']),
          'brillo_min': float(perc.loc['P2,5', 'brillo']), 'brillo_max': float(perc.loc['P97,5', 'brillo'])}


def motivos(m):
    # Devuelve la lista de criterios que la imagen incumple.
    r = []
    if m['nitidez'] < UMBRAL['nitidez_min']: r.append('nitidez')
    if m['contraste'] < UMBRAL['contraste_min']: r.append('contraste')
    if m['brillo'] < UMBRAL['brillo_min']: r.append('oscura')
    if m['brillo'] > UMBRAL['brillo_max']: r.append('clara')
    return r


cal['motivos'] = [','.join(motivos(m)) for m in cal[['brillo', 'contraste', 'nitidez']].to_dict('records')]
marcadas = cal[cal['motivos'] != '']
print(f'\nUmbrales: {UMBRAL}')
print(f'Imágenes marcadas por la regla: {len(marcadas)} de {len(cal)} ({len(marcadas) / len(cal) * 100:.1f} %)')
print(marcadas['motivos'].str.split(',').explode().value_counts().to_string())

VALORES['calidad_dataset_palma'] = {
    'lado_mayor_px': LADO_CAL,
    'percentiles': {m: {p: round(float(perc.loc[p, m]), 2) for p in perc.index} for m in perc.columns},
    'minimo': {m: round(float(cal[m].min()), 2) for m in METRICAS},
    'maximo': {m: round(float(cal[m].max()), 2) for m in METRICAS},
    'mediana_por_clase': {c: {m: round(float(por_clase_cal.loc[c, m]), 1) for m in METRICAS} for c in por_clase_cal.index},
    'umbrales_regla': {k: round(v, 1) for k, v in UMBRAL.items()},
    'imagenes_marcadas': int(len(marcadas)),
    'pct_marcadas': round(len(marcadas) / len(cal) * 100, 1),
    'marcadas_por_motivo': {k: int(v) for k, v in marcadas['motivos'].str.split(',').explode().value_counts().items()},
    'imagenes_con_mas_de_1pct_saturado': int((cal['pct_saturados'] > 1).sum()),
    'imagenes_con_mas_de_1pct_negro': int((cal['pct_oscuros'] > 1).sum()),
    'correlacion_nitidez_contraste': round(float(np.corrcoef(np.log10(cal['nitidez'].clip(lower=1e-3)), cal['contraste'])[0, 1]), 2),
}

### ¿Detecta la regla las degradaciones que más dañan al modelo?

Se aplica cada degradación a las imágenes de prueba y se cuenta qué porcentaje quedaría marcado por la regla. El resultado se cruza con el F1 macro medido en NB07.

In [ ]:
prueba = cal[cal['split'] == 'test'] if (cal['split'] == 'test').any() else cal
filas_regla = []
base_m = [metricas_calidad(IMG_CAL[r]) for r in prueba['ruta']]
filas_regla.append({'condicion': 'Sin degradar', 'nivel': '', 'parametro': '',
                    'nitidez_mediana': float(np.median([m['nitidez'] for m in base_m])),
                    'contraste_mediana': float(np.median([m['contraste'] for m in base_m])),
                    'brillo_mediana': float(np.median([m['brillo'] for m in base_m])),
                    'pct_marcadas': float(np.mean([len(motivos(m)) > 0 for m in base_m]) * 100),
                    'f1_macro_NB07': F1_NB07.get(('limpia', ''))})
for t in orden_deg:
    for nivel, valor in zip(('moderado', 'severo'), PARAM_DEG[t]):
        ms = [metricas_calidad(degradar(IMG_CAL[r], t, valor, semilla=i)) for i, r in enumerate(prueba['ruta'])]
        filas_regla.append({'condicion': NOMBRE_DEG[t], 'nivel': nivel, 'parametro': texto_param(t, valor),
                            'nitidez_mediana': float(np.median([m['nitidez'] for m in ms])),
                            'contraste_mediana': float(np.median([m['contraste'] for m in ms])),
                            'brillo_mediana': float(np.median([m['brillo'] for m in ms])),
                            'pct_marcadas': float(np.mean([len(motivos(m)) > 0 for m in ms]) * 100),
                            'f1_macro_NB07': F1_NB07.get((t, nivel))})
regla = pd.DataFrame(filas_regla)
regla.round(3).to_csv(DIR_VAL / 'tabla_3_regla_frente_a_degradaciones.csv', index=False)
print(f'Imágenes de prueba: {len(prueba)}')
print(regla.round(1).to_string(index=False))
VALORES['regla_frente_a_degradaciones'] = {'n_imagenes': int(len(prueba)),
    'filas': json.loads(regla.round(2).to_json(orient='records', force_ascii=False))}

### Figura 3.2 · Mejora del contraste en la imagen más oscura del conjunto

In [ ]:
fila_osc = cal.sort_values('brillo').iloc[0]
img_osc = IMG_CAL[fila_osc['ruta']]


def sobre_L(rgb, fn):
    # Aplica fn al canal de luminosidad y conserva el color.
    lab8 = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab8[:, :, 0] = fn(lab8[:, :, 0])
    return cv2.cvtColor(lab8, cv2.COLOR_LAB2RGB)


def estirar(L):
    p1, p99 = np.percentile(L, [1, 99])
    return np.clip((L.astype(np.float32) - p1) * 255.0 / max(p99 - p1, 1), 0, 255).astype(np.uint8)


clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
versiones = [('Original', img_osc),
             ('Estiramiento lineal', sobre_L(img_osc, estirar)),
             ('Ecualización global', sobre_L(img_osc, cv2.equalizeHist)),
             ('CLAHE', sobre_L(img_osc, clahe.apply))]
prop_o = img_osc.shape[0] / img_osc.shape[1]
fig, axs = plt.subplots(2, 4, figsize=(ANCHO_IN, ANCHO_IN * prop_o / 4 + ANCHO_IN * 0.17 + 0.55), constrained_layout=True,
                        gridspec_kw={'height_ratios': [prop_o, 0.62]})
info_contraste = {}
tope_h = 0
for k, (nombre, v) in enumerate(versiones):
    axs[0, k].imshow(v); sin_ejes(axs[0, k]); axs[0, k].set_title(f'({"abcd"[k]}) {nombre}')
    Lv = cv2.cvtColor(v, cv2.COLOR_RGB2LAB)[:, :, 0]
    hL = np.bincount(Lv.ravel(), minlength=256) / Lv.size
    axs[1, k].fill_between(np.arange(256), hL, step='mid', color=ACENTO, linewidth=0)
    tope_h = max(tope_h, float(np.sort(hL)[-3]))
    axs[1, k].set_xlim(0, 255); axs[1, k].set_xticks([0, 128, 255])
    axs[1, k].set_xlabel('Luminosidad (0–255)')
    axs[1, k].grid(axis='y', color=REJILLA, linewidth=0.5); axs[1, k].set_axisbelow(True)
    m = metricas_calidad(v)
    info_contraste[nombre] = {q: round(m[q], 1) for q in ('brillo', 'contraste', 'entropia', 'nitidez')}
for k in range(4):
    axs[1, k].set_ylim(0, tope_h * 1.25); eje_con_coma(axs[1, k].yaxis, 3)
    if k > 0:
        axs[1, k].set_yticklabels([])
axs[1, 0].set_ylabel('Proporción de píxeles')
guardar(fig, DIR_FIG3 / 'Fig_3_2_mejora_contraste.png')
plt.show()
VALORES['figura_3_2'] = {'imagen': fila_osc['archivo'], 'clase': NOMBRE_ES[fila_osc['clase']], 'metricas': info_contraste}
print(fila_osc['archivo'], json.dumps(info_contraste, ensure_ascii=False))

### Figura 3.3 · Ruido y filtros de suavizado

In [ ]:
def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return float('inf') if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)


lado_rec = min(280, min(img_ej_full.shape[:2]))
y0r = (img_ej_full.shape[0] - lado_rec) // 2; x0r = (img_ej_full.shape[1] - lado_rec) // 2
limpia = img_ej_full[y0r:y0r + lado_rec, x0r:x0r + lado_rec]
ruidosa = degradar(limpia, 'ruido', PARAM_DEG['ruido'][1], semilla=SEMILLA)
filtros = [('Original', limpia),
           (f'Con ruido (σ = {PARAM_DEG["ruido"][1]})', ruidosa),
           ('Filtro de media 5 × 5', cv2.blur(ruidosa, (5, 5))),
           ('Filtro gaussiano (σ = 1,5)', cv2.GaussianBlur(ruidosa, (0, 0), 1.5)),
           ('Filtro de mediana 5 × 5', cv2.medianBlur(ruidosa, 5)),
           ('Filtro bilateral', cv2.bilateralFilter(ruidosa, 9, 75, 7))]
fig, axs = plt.subplots(2, 3, figsize=(ANCHO_IN, ANCHO_IN * 2 / 3 + 0.55), constrained_layout=True)
info_filtros = {}
for ax, letra, (nombre, v) in zip(axs.ravel(), 'abcdef', filtros):
    ax.imshow(v, interpolation='nearest'); sin_ejes(ax)
    if nombre == 'Original':
        ax.set_title(f'({letra}) {nombre}\n(recorte de {lado_rec} × {lado_rec} px)', linespacing=1.15)
    else:
        valor_psnr = psnr(limpia, v)
        info_filtros[nombre] = {'psnr_dB': round(valor_psnr, 2), 'nitidez': round(metricas_calidad(v)['nitidez'], 1)}
        ax.set_title(f'({letra}) {nombre}\nPSNR = {coma(valor_psnr, 1)} dB', linespacing=1.15)
guardar(fig, DIR_FIG3 / 'Fig_3_3_ruido_filtros.png')
plt.show()
info_filtros['Original'] = {'nitidez': round(metricas_calidad(limpia)['nitidez'], 1)}
VALORES['figura_3_3'] = {'recorte_px': int(lado_rec), 'resultados': info_filtros}
print(json.dumps(info_filtros, ensure_ascii=False, indent=1))

### Figura 3.4 · La nitidez medida con el laplaciano

In [ ]:
base_nit = recorte_cuadrado(img_ej_full, 384)
sigmas = [0, 1.0, 2.5, 5.0]
fig, axs = plt.subplots(2, 4, figsize=(ANCHO_IN, ANCHO_IN / 2 + 0.6), constrained_layout=True)
info_nit = {}
tope_lap = None
for k, s in enumerate(sigmas):
    v = base_nit if s == 0 else cv2.GaussianBlur(base_nit, (0, 0), s)
    lap = cv2.Laplacian(cv2.cvtColor(v, cv2.COLOR_RGB2GRAY), cv2.CV_64F)
    if tope_lap is None:
        tope_lap = float(np.percentile(np.abs(lap), 99))
    axs[0, k].imshow(v); sin_ejes(axs[0, k])
    axs[0, k].set_title(f'({"abcd"[k]}) ' + ('Sin desenfoque' if s == 0 else f'σ = {coma(s, 1)} px'))
    axs[1, k].imshow(np.abs(lap), cmap='gray', vmin=0, vmax=tope_lap); sin_ejes(axs[1, k])
    axs[1, k].set_title(f'Varianza = {miles(lap.var()) if lap.var() >= 100 else coma(lap.var(), 1)}')
    info_nit[str(s)] = round(float(lap.var()), 1)
rotulo_fila(axs[0, 0], 'Imagen'); rotulo_fila(axs[1, 0], '|Laplaciano|')
guardar(fig, DIR_FIG3 / 'Fig_3_4_nitidez_laplaciano.png')
plt.show()
VALORES['figura_3_4'] = {'varianza_laplaciano_por_sigma': info_nit}
print(info_nit)

### Figura 3.5 · Distribución de las cuatro métricas en el conjunto de palma

In [ ]:
ETQ_MET = {'brillo': 'Brillo (media de gris, 0–255)', 'contraste': 'Contraste (desviación estándar)',
           'entropia': 'Entropía (bits)', 'nitidez': 'Nitidez (varianza del laplaciano)'}
fig, axs = plt.subplots(2, 2, figsize=(ANCHO_IN, ANCHO_IN * 0.62), constrained_layout=True)
for ax, letra, met in zip(axs.ravel(), 'abcd', METRICAS):
    x = cal[met].to_numpy()
    if met == 'nitidez':
        cajas = np.logspace(np.log10(max(x.min(), 1e-3)), np.log10(x.max()), 31)
        ax.set_xscale('log')
    else:
        cajas = np.linspace(x.min(), x.max(), 31)
    ax.hist(x, bins=cajas, color=ACENTO, edgecolor='white', linewidth=0.3)
    tope_y = ax.get_ylim()[1] * 1.22
    ax.set_ylim(0, tope_y)
    for p_n, p_v in (('P5', np.percentile(x, 5)), ('P50', np.percentile(x, 50)), ('P95', np.percentile(x, 95))):
        ax.axvline(p_v, color=TINTA, linewidth=0.8, linestyle='--' if p_n != 'P50' else '-')
        ax.text(p_v, tope_y * 0.985, p_n, ha='center', va='top', fontsize=6.3, color=TINTA,
                bbox=dict(facecolor='white', edgecolor='none', pad=1.2))
    ax.set_title(f'({letra}) {ETQ_MET[met]}', loc='left')
    ax.set_ylabel('Imágenes'); ax.grid(axis='y', color=REJILLA, linewidth=0.5); ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    if met == 'nitidez':
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: miles(v) if v >= 1 else coma(v, 1)))
        ax.xaxis.set_minor_formatter(FuncFormatter(lambda v, _: ''))
    elif met == 'entropia':
        eje_con_coma(ax.xaxis, 1)
guardar(fig, DIR_FIG3 / 'Fig_3_5_distribucion_calidad.png')
plt.show()

### Figura 3.6 · Los extremos del conjunto

In [ ]:
filas_ext = [('brillo', 'Brillo'), ('contraste', 'Contraste'), ('nitidez', 'Nitidez')]
fig = plt.figure(figsize=(ANCHO_IN, ANCHO_IN * 0.75 + 0.95), constrained_layout=True)
gs_ext = fig.add_gridspec(4, 5, height_ratios=[0.07, 1, 1, 1], width_ratios=[1, 1, 0.06, 1, 1])
info_ext = {}
for lado, col0, titulo in (('bajo', 0, 'Valores más bajos'), ('alto', 3, 'Valores más altos')):
    ax_t = fig.add_subplot(gs_ext[0, col0:col0 + 2]); ax_t.axis('off')
    ax_t.text(0.5, 0.5, titulo, ha='center', va='center', fontsize=8, fontweight='bold', transform=ax_t.transAxes)
    for f, (met, etq) in enumerate(filas_ext):
        orden_m = cal.sort_values(met, ascending=(lado == 'bajo')).head(2)
        info_ext[f'{met}_{lado}'] = [{'archivo': r['archivo'], 'clase': NOMBRE_ES[r['clase']], 'valor': round(float(r[met]), 1)}
                                     for _, r in orden_m.iterrows()]
        for k in range(2):
            ax = fig.add_subplot(gs_ext[f + 1, col0 + k]); sin_ejes(ax)
            if k < len(orden_m):
                r = orden_m.iloc[k]
                ax.imshow(recorte_cuadrado(IMG_CAL[r['ruta']], 360))
                valor_txt = miles(r[met]) if met == 'nitidez' and r[met] >= 100 else coma(r[met], 1)
                ax.set_title(f'{etq} = {valor_txt}', fontsize=7)
            if lado == 'bajo' and k == 0:
                rotulo_fila(ax, etq)
guardar(fig, DIR_FIG3 / 'Fig_3_6_extremos_calidad.png')
plt.show()
VALORES['figura_3_6'] = info_ext

### Figura 3.7 · Ajuste de tamaño y aumento de datos

In [ ]:
LADO_MODELO = 384          # tamaño de entrada del clasificador de palma
h_e, w_e = img_ej.shape[:2]


def con_relleno(img, lado, gris=114):
    esc = lado / max(img.shape[:2])
    red = cv2.resize(img, (round(img.shape[1] * esc), round(img.shape[0] * esc)), interpolation=cv2.INTER_AREA)
    lienzo = np.full((lado, lado, 3), gris, np.uint8)
    yy = (lado - red.shape[0]) // 2; xx = (lado - red.shape[1]) // 2
    lienzo[yy:yy + red.shape[0], xx:xx + red.shape[1]] = red
    return lienzo


cuadrada = recorte_cuadrado(img_ej, LADO_MODELO)
hsv_e = cv2.cvtColor(cuadrada, cv2.COLOR_RGB2HSV).astype(np.float32)
hsv_e[:, :, 0] = (hsv_e[:, :, 0] + 6) % 180; hsv_e[:, :, 1] = np.clip(hsv_e[:, :, 1] * 1.4, 0, 255)
M_rot = cv2.getRotationMatrix2D((LADO_MODELO / 2, LADO_MODELO / 2), 15, 1.0)
m_rec = min(h_e, w_e)
y_rec = int((h_e - m_rec) / 2 + 0.05 * m_rec); x_rec = int((w_e - m_rec) / 2 + 0.22 * m_rec)
recorte_al = cv2.resize(img_ej[y_rec:y_rec + int(0.62 * m_rec), x_rec:x_rec + int(0.62 * m_rec)],
                        (LADO_MODELO, LADO_MODELO), interpolation=cv2.INTER_AREA)
borrada = cuadrada.copy()
borrada[int(0.27 * LADO_MODELO):int(0.58 * LADO_MODELO), int(0.42 * LADO_MODELO):int(0.78 * LADO_MODELO)] = 114
aumentos = [
    (f'Original\n{img_ej_full.shape[1]} × {img_ej_full.shape[0]} px', img_ej_full),
    (f'Estirar a\n{LADO_MODELO} × {LADO_MODELO}', cv2.resize(img_ej, (LADO_MODELO, LADO_MODELO), interpolation=cv2.INTER_AREA)),
    ('Recorte central\ny reducción', cuadrada),
    ('Reducción\ncon relleno', con_relleno(img_ej, LADO_MODELO)),
    ('Volteo horizontal', cv2.flip(cuadrada, 1)),
    ('Volteo vertical', cv2.flip(cuadrada, 0)),
    ('Rotación de 15°', cv2.warpAffine(cuadrada, M_rot, (LADO_MODELO, LADO_MODELO), borderMode=cv2.BORDER_REFLECT_101)),
    ('Recorte aleatorio\ny cambio de escala', recorte_al),
    ('Brillo y contraste', degradar(degradar(cuadrada, 'contraste', 1.25), 'oscurecimiento', 0.8)),
    ('Tono y saturación', cv2.cvtColor(hsv_e.astype(np.uint8), cv2.COLOR_HSV2RGB)),
    ('Desenfoque leve', cv2.GaussianBlur(cuadrada, (0, 0), 1.5)),
    ('Borrado aleatorio', borrada),
]
fig = plt.figure(figsize=(ANCHO_IN, ANCHO_IN * 0.75 + 1.45), constrained_layout=True)
sub_au = fig.subfigures(2, 1, height_ratios=[1.12, 2], hspace=0.04)
sub_au[0].suptitle('Ajuste de tamaño a la entrada del modelo', x=0.01, ha='left', fontsize=8, fontweight='bold')
sub_au[1].suptitle('Aumento de datos: variantes de una misma imagen', x=0.01, ha='left', fontsize=8, fontweight='bold')
ejes_au = list(sub_au[0].subplots(1, 4)) + list(sub_au[1].subplots(2, 4).ravel())
for ax, letra, (nombre, v) in zip(ejes_au, 'abcdefghijkl', aumentos):
    ax.imshow(v); sin_ejes(ax); ax.set_title(f'({letra}) {nombre}', fontsize=7, linespacing=1.15)
    ax.set_anchor('N')
guardar(fig, DIR_FIG3 / 'Fig_3_7_tamano_aumento.png')
plt.show()

## 9. Guardar los valores para el texto

In [ ]:
VALORES['figuras_generadas'] = [Path(f).name for f in FIGURAS]
ruta_json = DIR_VAL / 'valores_cap03.json'
with open(ruta_json, 'w', encoding='utf-8') as fh:
    json.dump(VALORES, fh, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

print('Figuras generadas:', len(FIGURAS))
for f in FIGURAS:
    print('  ', Path(f).relative_to(DIR_LIBRO))
print('\nValores guardados en:', ruta_json.relative_to(DIR_LIBRO))
print('\nSiguiente paso: ejecute NB_LIBRO_03_Figuras_Capitulo_4.')